# CheMeleon — SMILES Collection & CoT Pre-Generation

Collects all unique SMILES from the 58 CheMeleon benchmarks (28 Polaris + 30 MoleculeACE),
then pre-generates **CoT texts** and **embeddings** for all molecules using the 9 general MMF queries.

### Cache Strategy
- **Per-query separation**: Each of the 9 queries gets its own CoT text file and embedding file
- This allows later selection of task-relevant queries (e.g., only `structure` + `solubility` for solubility tasks)
- **Naming**: `chemeleon_{query_key}_cot_texts.json` / `chemeleon_{query_key}_text_embeddings_compact.npz`
- **Storage**: float16 compressed npz (~2x smaller than float32)

### The 9 General Queries
| # | Key | Used by |
|---|-----|---------|
| 1 | `structure` | all tasks |
| 2 | `physical_properties` | solubility, binding |
| 3 | `solubility` | solubility |
| 4 | `reactivity` | all tasks |
| 5 | `optical_electrical` | quantum |
| 6 | `chirality` | quantum |
| 7 | `safety` | toxicity |
| 8 | `environmental` | toxicity |
| 9 | `applications` | binding |

In [1]:
# === Setup ===
import sys
import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

workspace_root = Path.cwd().parent.parent  # benchmarking/chemeleon/ → workspace root
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

COT_TEXT_DIR = workspace_root / "cache" / "cot_texts"
COT_EMB_DIR  = workspace_root / "cache" / "cot_embeddings"
SMILES_PATH  = workspace_root / "cache" / "chemeleon_smiles.json"

COT_TEXT_DIR.mkdir(parents=True, exist_ok=True)
COT_EMB_DIR.mkdir(parents=True, exist_ok=True)

# The 9 general MMF queries (union of all TASK_QUERY_SUBSETS)
QUERY_KEYS = [
    "structure",
    "physical_properties",
    "solubility",
    "reactivity",
    "optical_electrical",
    "chirality",
    "safety",
    "environmental",
    "applications",
]

print(f"Workspace: {workspace_root}")
print(f"CoT texts:     {COT_TEXT_DIR}")
print(f"CoT embeddings: {COT_EMB_DIR}")
print(f"Queries: {len(QUERY_KEYS)}")

Workspace: c:\Users\U10154695\Dev\Prediction\MPP PoC
CoT texts:     c:\Users\U10154695\Dev\Prediction\MPP PoC\cache\cot_texts
CoT embeddings: c:\Users\U10154695\Dev\Prediction\MPP PoC\cache\cot_embeddings
Queries: 9


In [2]:
# === Benchmark Definitions (same as in the evaluate.py scripts) ===

POLARIS_BENCHMARKS = (
    "polaris/pkis2-ret-wt-cls-v2",
    "polaris/pkis2-ret-wt-reg-v2",
    "polaris/pkis2-kit-wt-cls-v2",
    "polaris/pkis2-kit-wt-reg-v2",
    "polaris/pkis2-egfr-wt-reg-v2",
    "polaris/adme-fang-solu-1",
    "polaris/adme-fang-rppb-1",
    "polaris/adme-fang-hppb-1",
    "polaris/adme-fang-perm-1",
    "polaris/adme-fang-rclint-1",
    "polaris/adme-fang-hclint-1",
    "tdcommons/lipophilicity-astrazeneca",
    "tdcommons/ppbr-az",
    "tdcommons/clearance-hepatocyte-az",
    "tdcommons/cyp2d6-substrate-carbonmangels",
    "tdcommons/half-life-obach",
    "tdcommons/cyp2c9-substrate-carbonmangels",
    "tdcommons/clearance-microsome-az",
    "tdcommons/dili",
    "tdcommons/bioavailability-ma",
    "tdcommons/vdss-lombardo",
    "tdcommons/cyp3a4-substrate-carbonmangels",
    "tdcommons/pgp-broccatelli",
    "tdcommons/caco2-wang",
    "tdcommons/herg",
    "tdcommons/bbb-martins",
    "tdcommons/ames",
    "tdcommons/ld50-zhu",
)

MOLECULEACE_BENCHMARKS = (
    "CHEMBL1862_Ki",  "CHEMBL1871_Ki",  "CHEMBL2034_Ki",  "CHEMBL2047_EC50",
    "CHEMBL204_Ki",   "CHEMBL2147_Ki",  "CHEMBL214_Ki",   "CHEMBL218_EC50",
    "CHEMBL219_Ki",   "CHEMBL228_Ki",   "CHEMBL231_Ki",   "CHEMBL233_Ki",
    "CHEMBL234_Ki",   "CHEMBL235_EC50", "CHEMBL236_Ki",   "CHEMBL237_EC50",
    "CHEMBL237_Ki",   "CHEMBL238_Ki",   "CHEMBL239_EC50", "CHEMBL244_Ki",
    "CHEMBL262_Ki",   "CHEMBL264_Ki",   "CHEMBL2835_Ki",  "CHEMBL287_Ki",
    "CHEMBL2971_Ki",  "CHEMBL3979_EC50","CHEMBL4005_Ki",  "CHEMBL4203_Ki",
    "CHEMBL4616_EC50","CHEMBL4792_Ki",
)

MOLECULEACE_CSV_BASE = (
    "https://raw.githubusercontent.com/molML/MoleculeACE/"
    "7e6de0bd2968c56589c580f2a397f01c531ede26/"
    "MoleculeACE/Data/benchmark_data"
)

print(f"Polaris:     {len(POLARIS_BENCHMARKS)} benchmarks")
print(f"MoleculeACE: {len(MOLECULEACE_BENCHMARKS)} benchmarks")
print(f"Total:       {len(POLARIS_BENCHMARKS) + len(MOLECULEACE_BENCHMARKS)}")

Polaris:     28 benchmarks
MoleculeACE: 30 benchmarks
Total:       58


In [3]:
# === Collect SMILES ===

# --- Polaris: SKIPPED (polaris-lib has SSL/streaming issues behind corporate proxy) ---
# TODO: Run from outside the corporate network or patch polaris httpx client.
#       Then uncomment and use po.load_benchmark() to collect SMILES.
polaris_smiles: dict[str, set[str]] = {}
polaris_all: set[str] = set()
print("Polaris benchmarks: SKIPPED (corporate proxy)\n")

# --- MoleculeACE: load from GitHub CSVs ---
mace_smiles: dict[str, set[str]] = {}
mace_errors: list[str] = []

for name in MOLECULEACE_BENCHMARKS:
    try:
        url = f"{MOLECULEACE_CSV_BASE}/{name}.csv"
        df = pd.read_csv(url)
        smiles_set = set(df["smiles"])
        mace_smiles[name] = smiles_set
        n_train = (df["split"] == "train").sum()
        n_test  = (df["split"] == "test").sum()
        print(f"  {name:25s}  train={n_train:>5}  test={n_test:>5}  unique={len(smiles_set):>5}")
    except Exception as e:
        mace_errors.append(f"{name}: {e}")
        print(f"  {name:25s}  ERROR: {e}")

mace_all = set().union(*mace_smiles.values()) if mace_smiles else set()
all_smiles = polaris_all | mace_all

print(f"\nMoleculeACE unique: {len(mace_all):,}")
print(f"Combined unique:    {len(all_smiles):,}")
if mace_errors:
    print(f"Errors: {len(mace_errors)}")

Polaris benchmarks: SKIPPED (corporate proxy)

  CHEMBL1862_Ki              train=  633  test=  161  unique=  794
  CHEMBL1871_Ki              train=  525  test=  134  unique=  659
  CHEMBL2034_Ki              train=  598  test=  152  unique=  750
  CHEMBL2047_EC50            train=  503  test=  128  unique=  631
  CHEMBL204_Ki               train= 2201  test=  553  unique= 2754
  CHEMBL2147_Ki              train= 1162  test=  294  unique= 1456
  CHEMBL214_Ki               train= 2651  test=  666  unique= 3317
  CHEMBL218_EC50             train=  823  test=  208  unique= 1031
  CHEMBL219_Ki               train= 1491  test=  374  unique= 1865
  CHEMBL228_Ki               train= 1362  test=  342  unique= 1704
  CHEMBL231_Ki               train=  776  test=  197  unique=  973
  CHEMBL233_Ki               train= 2512  test=  630  unique= 3142
  CHEMBL234_Ki               train= 2924  test=  733  unique= 3657
  CHEMBL235_EC50             train= 1877  test=  472  unique= 2349
  CHEMBL236_Ki 

In [4]:
# === Save SMILES manifest ===

smiles_sorted = sorted(all_smiles)

output = {
    "metadata": {
        "n_polaris_benchmarks": len(POLARIS_BENCHMARKS),
        "n_moleculeace_benchmarks": len(MOLECULEACE_BENCHMARKS),
        "n_polaris_unique": len(polaris_all),
        "n_moleculeace_unique": len(mace_all),
        "n_total_unique": len(all_smiles),
    },
    "per_benchmark": {
        name: sorted(sset) for name, sset in {**polaris_smiles, **mace_smiles}.items()
    },
    "all_unique_smiles": smiles_sorted,
}

with open(SMILES_PATH, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(smiles_sorted):,} unique SMILES to {SMILES_PATH.name}")
print(f"File size: {SMILES_PATH.stat().st_size / 1024:.1f} KB")

# Per-benchmark sizes
sizes = {k: len(v) for k, v in {**polaris_smiles, **mace_smiles}.items()}
sizes_df = pd.DataFrame.from_dict(sizes, orient="index", columns=["unique_smiles"])
sizes_df = sizes_df.sort_values("unique_smiles", ascending=False)
print(f"\n{sizes_df.to_string()}")
print(f"\nEstimated API effort: {len(all_smiles)} SMILES × {len(QUERY_KEYS)} queries = {len(all_smiles) * len(QUERY_KEYS):,} LLM calls")

Saved 35,633 unique SMILES to chemeleon_smiles.json
File size: 5508.5 KB

                 unique_smiles
CHEMBL234_Ki              3657
CHEMBL214_Ki              3317
CHEMBL233_Ki              3142
CHEMBL244_Ki              3097
CHEMBL264_Ki              2862
CHEMBL204_Ki              2754
CHEMBL237_Ki              2603
CHEMBL236_Ki              2598
CHEMBL235_EC50            2349
CHEMBL219_Ki              1865
CHEMBL239_EC50            1721
CHEMBL228_Ki              1704
CHEMBL4792_Ki             1471
CHEMBL2147_Ki             1456
CHEMBL287_Ki              1328
CHEMBL3979_EC50           1125
CHEMBL238_Ki              1052
CHEMBL218_EC50            1031
CHEMBL2971_Ki              976
CHEMBL231_Ki               973
CHEMBL4005_Ki              960
CHEMBL237_EC50             955
CHEMBL262_Ki               856
CHEMBL1862_Ki              794
CHEMBL2034_Ki              750
CHEMBL4203_Ki              731
CHEMBL4616_EC50            682
CHEMBL1871_Ki              659
CHEMBL2047_EC50            

In [5]:
# === Azure OpenAI Client ===
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv(workspace_root / ".env")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview"),
)

COT_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5-chat")
EMB_MODEL = "text-embedding-3-large"

# Quick connectivity check (max_completion_tokens, not max_tokens — required by newer models)
resp = client.chat.completions.create(
    model=COT_MODEL,
    messages=[{"role": "user", "content": "Say OK"}],
    max_completion_tokens=5,
)
print(f"Model: {COT_MODEL}  →  {resp.choices[0].message.content}")
print(f"Embedding model: {EMB_MODEL}")
print(f"SMILES to process: {len(smiles_sorted):,}")
print(f"Queries: {len(QUERY_KEYS)} → {len(smiles_sorted) * len(QUERY_KEYS):,} LLM calls total")

Model: gpt-5.2-chat  →  
Embedding model: text-embedding-3-large
SMILES to process: 35,633
Queries: 9 → 320,697 LLM calls total


In [ ]:
# === Generate CoT Texts (9 queries × all SMILES, parallel) ===
#
# Each query produces a separate cache file:
#   cache/cot_texts/chemeleon_{query_key}_cot_texts.json
#
# The generator auto-resumes from cache, so re-running is safe.
# NOTE: temperature is left as None (default) because reasoning models
#       (o-series / gpt-5) reject temperature != 1.0.

from prompts.cot_generator import CoTGenerator, CoTConfig, MMF_SYSTEM_PROMPT, MMF_QUERY_TEMPLATES

MAX_WORKERS = 1024  # parallel LLM threads per query
SAVE_INTERVAL = 100  # flush cache every N molecules
WARMUP_N = 3  # sequential calls before going parallel (avoids cold-start throttling)

for i, qkey in enumerate(QUERY_KEYS, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(QUERY_KEYS)}] Query: {qkey}")
    print(f"{'='*60}")

    config = CoTConfig(
        task_name=f"chemeleon_{qkey}",
        system_prompt=MMF_SYSTEM_PROMPT,
        user_prompt_template=MMF_QUERY_TEMPLATES[qkey],
        cot_model=COT_MODEL,
        embedding_model=EMB_MODEL,
        multi_query_mode=False,   # one query per generator
        cache_dir=COT_TEXT_DIR,
        batch_size=50,
        # temperature=None by default → not sent to API (reasoning models require this)
    )
    gen = CoTGenerator(config=config)

    # Warmup: run a few sequential calls to establish connection pool / token
    warmup_done = gen.get_cot_texts(
        smiles_list=smiles_sorted[:WARMUP_N],
        client=client,
        verbose=False,
    )
    print(f"  Warmup: {len(warmup_done)} sequential calls OK")

    # Full parallel generation (skips already-cached from warmup + prior runs)
    cot_texts = gen.get_cot_texts_parallel(
        smiles_list=smiles_sorted,
        client=client,
        max_workers=MAX_WORKERS,
        verbose=True,
        save_interval=SAVE_INTERVAL,
    )
    print(f"  → {len(cot_texts):,} texts cached at {config.task_name}_cot_texts.json")

print(f"\n✓ All {len(QUERY_KEYS)} query caches written to {COT_TEXT_DIR}")


[1/9] Query: structure
  Warmup: 3 sequential calls OK
CoT texts: 3 cached, 35630 to generate (1024 workers)


Generating CoT texts (parallel):  17%|█▋        | 5962/35630 [03:20<2:52:48,  2.86it/s]

In [ ]:
# === Generate Compact Embeddings (npz, float16) ===
#
# For each query, embed all CoT texts and save as:
#   cache/cot_embeddings/chemeleon_{query_key}_text_embeddings_compact.npz
#   cache/cot_embeddings/chemeleon_{query_key}_text_embeddings_index.json

from utils.embedding_cache import regenerate_compact_cache

BATCH_SIZE_EMB = 100  # texts per embedding API call

for i, qkey in enumerate(QUERY_KEYS, 1):
    cot_path = COT_TEXT_DIR / f"chemeleon_{qkey}_cot_texts.json"
    npz_path = COT_EMB_DIR / f"chemeleon_{qkey}_text_embeddings_compact.npz"

    if npz_path.exists():
        size_mb = npz_path.stat().st_size / (1024**2)
        print(f"[{i}/{len(QUERY_KEYS)}] {qkey}: SKIP (exists, {size_mb:.1f} MB)")
        continue

    if not cot_path.exists():
        print(f"[{i}/{len(QUERY_KEYS)}] {qkey}: SKIP (no CoT texts at {cot_path.name})")
        continue

    print(f"\n[{i}/{len(QUERY_KEYS)}] {qkey}")
    regenerate_compact_cache(
        cot_texts_path=cot_path,
        output_path=npz_path,
        client=client,
        embedding_model=EMB_MODEL,
        batch_size=BATCH_SIZE_EMB,
        dtype="float16",
    )

print(f"\n✓ All embedding caches in {COT_EMB_DIR}")

In [ ]:
# === Summary ===
import os as _os

print("=== Generated Files ===\n")

# SMILES manifest
if SMILES_PATH.exists():
    print(f"  {SMILES_PATH.name:50s} {SMILES_PATH.stat().st_size/1024:>8.1f} KB")

# CoT texts
total_texts_kb = 0
for qkey in QUERY_KEYS:
    p = COT_TEXT_DIR / f"chemeleon_{qkey}_cot_texts.json"
    if p.exists():
        size_kb = p.stat().st_size / 1024
        total_texts_kb += size_kb
        with open(p) as f:
            n = len(json.load(f))
        print(f"  cot_texts/chemeleon_{qkey:25s}_cot_texts.json  {size_kb:>8.1f} KB  ({n:,} mol)")
    else:
        print(f"  cot_texts/chemeleon_{qkey:25s}_cot_texts.json  MISSING")

# Embeddings
total_emb_mb = 0
for qkey in QUERY_KEYS:
    npz = COT_EMB_DIR / f"chemeleon_{qkey}_text_embeddings_compact.npz"
    idx = COT_EMB_DIR / f"chemeleon_{qkey}_text_embeddings_index.json"
    if npz.exists():
        size_mb = npz.stat().st_size / (1024**2)
        total_emb_mb += size_mb
        print(f"  cot_embeddings/chemeleon_{qkey:25s}_..._compact.npz   {size_mb:>6.1f} MB")
    else:
        print(f"  cot_embeddings/chemeleon_{qkey:25s}_..._compact.npz   MISSING")

print(f"\n  Total CoT texts:     {total_texts_kb/1024:>6.1f} MB")
print(f"  Total embeddings:    {total_emb_mb:>6.1f} MB")
print(f"  Grand total:         {total_texts_kb/1024 + total_emb_mb:>6.1f} MB")

## Next Steps

1. **Polaris**: Run SMILES collection from outside corporate network (reuse this notebook)
2. **`seg_evaluate.py`**: Write SEG evaluation script following CheMeleon pattern
   - Load pre-cached embeddings per query via `EfficientEmbeddingCache`
   - Select task-relevant queries per benchmark (see table in header)
   - Concatenate selected query embeddings as input features